<a href="https://colab.research.google.com/github/asaah262/lab-neural-networks/blob/master/challenge_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [52]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Read the dataset
df = pd.read_csv("/tic-tac-toe.csv")

# Inspect the dataset

print(df.head())
print(df.info())
print(df.describe(include="all"))
print(df['class'].value_counts())

# The dataset looks reliable because:
# - there are no missing values
# - the board values are consistent: x, o, b
# - the target column contains only True/False

# Convert categorical values to numeric
cell_map = {"x": 1, "o": -1, "b": 0}
for col in df.columns[:-1]:
    df[col] = df[col].map(cell_map)

df["class"] = df["class"].astype(int)   # False -> 0, True -> 1

# Separate inputs and output
X = df.drop("class", axis=1)
y = df["class"]

# Normalize the input data
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print("\nEncoded and normalized sample:")
print(X_scaled[:5])
print(y.head())

  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB
None
         TL   TM   TR   ML   MM   MR   BL   BM   BR class
count   958  958  958  958  958  958  958  958  958   958
unique    3    3    3    3    3    3    3    3    3     2
top       x    x  

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [53]:
# Split the training and test data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Create a Sequential model
model = tf.keras.Sequential([
    tf.keras.Input(shape=(9,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Fit training data
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

# Evaluate on test data
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print("\nInitial model results:")
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

# Save model
model.save("tic-tac-toe.model.keras")


Epoch 1/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.6585 - loss: 0.6224 - val_accuracy: 0.6364 - val_loss: 0.6317
Epoch 2/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6585 - loss: 0.5973 - val_accuracy: 0.6364 - val_loss: 0.6123
Epoch 3/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6601 - loss: 0.5713 - val_accuracy: 0.6883 - val_loss: 0.5826
Epoch 4/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7337 - loss: 0.5445 - val_accuracy: 0.7338 - val_loss: 0.5578
Epoch 5/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7680 - loss: 0.5237 - val_accuracy: 0.7468 - val_loss: 0.5376
Epoch 6/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7647 - loss: 0.5107 - val_accuracy: 0.7662 - val_loss: 0.5242
Epoch 7/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7859 - loss: 0.4926 - val_accuracy: 0.7727 - val_loss: 0.5158
Epoch 8/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7941 - loss: 0.4798 - val_accuracy: 0.7662 - val_loss

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [54]:
# Load the saved model
saved_model = load_model("tic-tac-toe.model.keras")

# Pick a few random rows from the test set
np.random.seed(42)
random_indices = np.random.choice(len(X_test), size=5, replace=False)

sample_X = X_test[random_indices]
sample_y = y_test.iloc[random_indices]

# Predict
pred_probs = saved_model.predict(sample_X)
pred_classes = np.argmax(pred_probs, axis=1)

print("\nPredictions on random test rows:")
for i in range(len(random_indices)):
    print(f"Row index in test set: {random_indices[i]}")
    print("Predicted:", pred_classes[i], "| Actual:", sample_y.iloc[i])
    print("Correct:", pred_classes[i] == sample_y.iloc[i])
    print("---")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step

Predictions on random test rows:
Row index in test set: 45
Predicted: 1 | Actual: 1
Correct: True
---
Row index in test set: 136
Predicted: 1 | Actual: 1
Correct: True
---
Row index in test set: 76
Predicted: 1 | Actual: 1
Correct: True
---
Row index in test set: 143
Predicted: 1 | Actual: 1
Correct: True
---
Row index in test set: 113
Predicted: 1 | Actual: 1
Correct: True
---


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [55]:
# 1. more layers
# 2. custom learning rate
# 3. more epochs

improved_model = tf.keras.Sequential([
    tf.keras.Input(shape=(9,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax")
])

custom_optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)

improved_model.compile(
    optimizer=custom_optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

improved_history = improved_model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

improved_loss, improved_accuracy = improved_model.evaluate(X_test, y_test, verbose=0)
print("\nImproved model results:")
print("Test loss:", improved_loss)
print("Test accuracy:", improved_accuracy)


Epoch 1/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.6683 - loss: 0.5948 - val_accuracy: 0.7662 - val_loss: 0.5554
Epoch 2/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7598 - loss: 0.5179 - val_accuracy: 0.7597 - val_loss: 0.5107
Epoch 3/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7925 - loss: 0.4350 - val_accuracy: 0.8117 - val_loss: 0.4165
Epoch 4/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8660 - loss: 0.3166 - val_accuracy: 0.9416 - val_loss: 0.2248
Epoch 5/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9526 - loss: 0.1907 - val_accuracy: 0.9156 - val_loss: 0.1512
Epoch 6/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9624 - loss: 0.1274 - val_accuracy: 0.9481 - val_loss: 0.1064
Epoch 7/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9690 - loss: 0.0987 - val_accuracy: 0.9805 - val_loss: 0.0694
Epoch 8/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9804 - loss: 0.0599 - val_accuracy: 0.9935 - 

**Which approach(es) did you find helpful to improve your model performance?**

The initial model already achieved high accuracy (97.39%) and low loss, indicating strong performance.
Attempts to improve the model by adding more layers, increasing epochs, and adjusting the learning rate did not improve accuracy and slightly increased the loss.

This suggests that the problem is relatively simple and does not require a more complex model.
The additional layers likely caused slight overfitting without improving generalization.

Therefore, the most effective approach was using a simpler model, as increasing complexity did not provide any benefit.S